# <center> AttentionNMT: GRU-Attention Encoder-Decoder Neural Machine Translation </center>
Project details are available on the GitHub repository: [AttentionNMT](https://github.com/Hoom4n/AttentionNMT)

# Configuration

In [1]:
import os, re, importlib, unicodedata, json
from itertools import chain

import pandas as pd
import tokenizers
from torchmetrics import Accuracy

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

pd.set_option("max_colwidth", 500)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch Device: {device}")

Torch Device: cuda


In [2]:
%%writefile src/config.py
from dataclasses import dataclass, field

@dataclass
class HPARAMS:
    vocab_size = 14_000
    max_seq_len = 32
    batch_size = 64
    
    model_hparams: dict = field(default_factory=lambda: {
    "embedding_dim" : 512,
    "hidden_dim" : 512,
    "gru_layers" : 2,
    "gru_dropout" : 0.1,
    "pad_token_id" : 0
    })
    
    optimizer_hparams: dict = field(default_factory=lambda: {
        "lr": 1e-3,
        "weight_decay": 3e-5
    })


    trainer_hparams: dict = field(default_factory=lambda: {
    "n_epochs": 15,
    "enable_mixed_precision": True,
    "restore_best_model" : False,
    "use_early_stopping" : True,
    "early_stopping_patience" : 3,
    "grad_clip_value" : 1.0
    })

Overwriting src/config.py


In [3]:
import src.config
importlib.reload(src.config)
from src.config import HPARAMS
hp = HPARAMS()

def create_project_dirs(root_dir: str,
                        dirs: list = ["data", "artifacts", "model", "tokenizer"]) -> list:
    """Create project directories and return their paths."""
    path_list = [os.path.join(root_dir, dir_) for dir_ in dirs]
    for p in path_list:
        os.makedirs(p, exist_ok=True)
    return path_list

ROOT_DIR = os.getcwd()
DATA_PATH, ARTIFACTS_PATH, MODEL_PATH, TOKENIZER_PATH = create_project_dirs(ROOT_DIR)

# Data Preparation

In [4]:
def quick_eda(data_path):
    df = pd.read_parquet(data_path)
    print(f"*** dataset size: {df.shape[0]:,} ***")
    print(f"*** nan count: {df.isna().sum().sum()} ***")
    print("*** word count describe: ***")
    print(pd.concat([df[col].str.split().str.len().describe() for col in df.columns], axis=1))
    print(f"*** uniques characters: ***\n{sorted(set("".join(chain(df.target_text , df.source_text))))}")
    return df.head(5)

quick_eda(os.path.join(DATA_PATH, "eng_spa.parquet"))

*** dataset size: 221,813 ***
*** nan count: 0 ***
*** word count describe: ***
         source_text    target_text
count  221813.000000  221813.000000
mean        6.917854       6.729673
std         4.070935       9.192844
min         1.000000       1.000000
25%         5.000000       4.000000
50%         6.000000       6.000000
75%         8.000000       8.000000
max       214.000000    3823.000000
*** uniques characters: ***
['\t', '\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '¡', '¨', 'ª', '«', '\xad', '°', '³', '´', 'º', '»', '¿', 'À', 'Á', 'Â', 'Ã', 'Ç', 'É', 'Í', 'Ñ', 'Ó', 'Ú', 'Ü', 'ß', 'à

,source_text,target_text
0,100 euros for the whole day.,100 euros por toda la jornada.
1,"100 per cent of us die, and the percentage cannot be increased.","El cien por cien de los seres humanos mueren, y este porcentaje no puede aumentar."
2,10 minutes remained until the end of the lesson.,Faltaban 10 minutos para el final de la clase.
3,10% of the inhabitants come from Japan.,10% de los habitantes provienen de Japón.
4,123456 is a frequently-used password.,123456 es una contraseña usada a menudo.


In [5]:
def get_data(data_path, val_frac = 0.15):
    df = pd.read_parquet(data_path)\
        .sample(frac=1.0, random_state=42)\
        .reset_index(drop=True)
    
    split = int(len(df) * val_frac)
    val, train = df.iloc[:split], df.iloc[split:]
    print(f"*** train size: {len(train):,} | val size: {len(val):,} ***")
    return train, val

train, val = get_data(os.path.join(DATA_PATH, "eng_spa.parquet"))

*** train size: 188,542 | val size: 33,271 ***


# Training BPE Tokenizer

In [6]:
def bpe_trainer(
    train_iterator,
    vocab_size,
    save_path,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
    enable_truncation = True,
    max_seq_len = 128,
    enable_padding = True,
    allowed_chars = r"[^a-z0-9áéíóúüñ¿¡\.\,\!\?\:\;\"\'\-\(\)\s]",
    sample_demo_text = "[BOS] I love Deep Learning!!! [EOS]"
):
    """
    Train and save a Byte-Pair Encoding (BPE) tokenizer.

    This function builds a subword tokenizer from a text iterator (e.g., training corpus),
    applies Unicode normalization, lowercasing, and character filtering, then trains a joint
    vocabulary for both source and target languages (ideal for bilingual NMT setups).
    It also configures truncation, padding, and outputs a sample encoding preview.
    """
    tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
    
    tokenizer.normalizer = tokenizers.normalizers.Sequence([
            tokenizers.normalizers.NFKC(), # unicode normalization
            tokenizers.normalizers.Lowercase(),
            tokenizers.normalizers.Replace(tokenizers.Regex(allowed_chars), ""), # keep only valid en-es charchters
            tokenizers.normalizers.Replace(tokenizers.Regex(r"\s+"), " "), # collapse runs of spaces
        ])

    trainer = tokenizers.trainers.BpeTrainer(
        vocab_size = vocab_size,
        special_tokens=special_tokens,
        show_progress = False
    )
    tokenizer.train_from_iterator(train_iterator, trainer)
    print(f"*** vocab size: {tokenizer.get_vocab_size():,} ***")

    if enable_truncation:
        tokenizer.enable_truncation(max_seq_len)
        print(f"*** tokenizer truncated to max len: {max_seq_len} ***")

    if enable_padding:
        tokenizer.enable_padding(pad_token="[PAD]", pad_id=0)

    if sample_demo_text is not None:
        enc = tokenizer.encode(sample_demo_text)
        print(f"\nSample demo text: {sample_demo_text}")
        print(f"Tokens: {enc.tokens}")
        print(f"Token IDs: {enc.ids}")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    tokenizer.save(save_path)
    print(f"\n*** trained tokenizer saved to: {save_path} ***")

bpe_trainer(chain(train.source_text, train.target_text),
            hp.vocab_size,
            os.path.join(TOKENIZER_PATH,"bpe_tokenizer.json"),
            max_seq_len=hp.max_seq_len)

*** vocab size: 14,000 ***
*** tokenizer truncated to max len: 32 ***

Sample demo text: [BOS] I love Deep Learning!!! [EOS]
Tokens: ['[BOS]', 'i', 'love', 'deep', 'learning', '!', '!', '!', '[EOS]']
Token IDs: [2, 33, 561, 3108, 2634, 4, 4, 4, 3]

*** trained tokenizer saved to: /mnt/c/Users/ASUS/Documents/Machine-Learning/GitHUB/AttentionNMT/tokenizer/bpe_tokenizer.json ***


# Dataset & Collate Func

In [7]:
class NMTDataset(Dataset):
    """Custom Dataset for paired source–target text sequences."""
    def __init__(self, src_iterator, tgt_iterator):
        assert len(src_iterator) == len(tgt_iterator), "src and target must have same size"
        self.src_iterator = src_iterator
        self.tgt_iterator = tgt_iterator

        if isinstance(self.src_iterator, pd.Series):
            self.src_iterator = self.src_iterator.reset_index(drop=True)
        if isinstance(self.tgt_iterator, pd.Series):
            self.tgt_iterator = self.tgt_iterator.reset_index(drop=True)
        
    def __len__(self):
        return len(self.src_iterator)

    def __getitem__(self, idx):
        return (self.src_iterator[idx] , self.tgt_iterator[idx])

In [8]:
def collate_fn(batch, pad_token_id = 0):
    src_list, tgt_list = zip(*batch)
    tgt_list = [f"[BOS] {s} [EOS]" for s in tgt_list]
    
    src_encodings = tokenizer.encode_batch(src_list)
    tgt_encodings = tokenizer.encode_batch(tgt_list)
    
    src_ids = torch.tensor([enc.ids for enc in src_encodings], dtype=torch.long)
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings], dtype=torch.long)
    tgt_ids = torch.tensor([enc.ids for enc in tgt_encodings], dtype=torch.long)
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings], dtype=torch.long)
    labels = tgt_ids[:, 1:].clone()
    # assign -100 to padding steps, making them invisible to the loss computation
    labels[labels == pad_token_id] = -100
    
    return ({
        "input_ids": src_ids, # encoder input
        "attention_mask": src_mask, # encoder mask
        "decoder_input_ids": tgt_ids[:, :-1], # decoder input
        "decoder_attention_mask": tgt_mask[:, :-1] # decoder mask
            }, labels)

<a id='section2'></a>

# Model

In [9]:
%%writefile src/model.py
import math
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Attention(nn.Module):
    """
    Scaled Dot Product Attention (Luong-Style) with Masking Support.
    
    Args:
        query: (N, Lq, Dq)
        key:   (N, Lk, Dk)
        value: (N, Lv, Dv), optional. If None, value=key
        mask:  (N, Lk), optional. 1 for valid tokens, 0 for padding.
    
    Returns:
        context: (N, Lq, Dv)
    """
    def __init__(self, use_scale=True):
        super().__init__()
        self.use_scale = use_scale
        
    def forward(self, query, key, value=None, mask=None):
        assert query.shape[-1] == key.shape[-1], "query & key must have same hidden dim"     
        if value is None:
            value = key
        else:
            assert key.shape[1] == value.shape[1], "key & value must have same sequence length"
        
        scale_factor = 1 / math.sqrt(key.size(-1)) if self.use_scale else 1 # 1 / sqrt(Dk)
        attention_scores = query @ key.transpose(-2, -1) * scale_factor  # (N, Lq, Lk)

        if mask is not None:
            mask = mask.unsqueeze(1)  # (N, 1, Lk)
            attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))
            
        attention_weights = torch.softmax(attention_scores, dim=-1)  # (N, Lq, Lk)
        return attention_weights @ value  # (N, Lq, Dv)

Overwriting src/model.py


In [10]:
%%writefile -a src/model.py

class NMTModel(nn.Module):
    """
    Neural Machine Translation (NMT) model with GRU encoder–decoder and Luong-style attention.

    Args:
        vocab_size (int): Size of the vocabulary.
        embedding_dim (int, optional): Dimension of token embeddings. Default: 512.
        hidden_dim (int, optional): Dimension of GRU hidden states. Default: 512.
        gru_layers (int, optional): Number of GRU layers for encoder and decoder. Default: 2.
        gru_dropout (float, optional): Dropout probability between GRU layers. Default: 0.1.
        pad_token_id (int, optional): Index of the padding token. Default: 0.

    Forward Inputs:
        input_ids (LongTensor): Source token IDs, shape (N, L_src).
        attention_mask (LongTensor): Source mask, shape (N, L_src), 1 for valid tokens.
        decoder_input_ids (LongTensor): Target token IDs, shape (N, L_tgt).
        decoder_attention_mask (LongTensor, optional): Target mask, shape (N, L_tgt).

    Forward Returns:
        logits (FloatTensor): Prediction scores, shape (N, vocab_size, L_tgt).
    """
    def __init__(self, vocab_size, embedding_dim = 512,
                 hidden_dim = 512, gru_layers = 2,
                 gru_dropout = 0.1, pad_token_id = 0):
        
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_token_id)
        self.encoder = nn.GRU(embedding_dim, hidden_dim,
                              num_layers = gru_layers, batch_first=True,
                              dropout = gru_dropout)
        self.decoder = nn.GRU(embedding_dim, hidden_dim,
                              num_layers = gru_layers, batch_first=True,
                              dropout = gru_dropout)
        self.attention = Attention()
        self.output = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, input_ids, attention_mask,
                decoder_input_ids, decoder_attention_mask=None):

        # source and target embedding
        src_embeddings = self.embedding(input_ids) # (N, L_src, emb_dim)
        tgt_embeddings = self.embedding(decoder_input_ids) # (N, L_tgt, emb_dim)

        # encoder
        src_lenghts = attention_mask.sum(dim=1)
        packed_encoder_inputs = pack_padded_sequence(src_embeddings,
                                                     src_lenghts.cpu(),
                                                     batch_first=True,
                                                     enforce_sorted=False)
        
        encoder_outputs_packed , encoder_hidden = self.encoder(packed_encoder_inputs) # encoder_hidden: (num_layers, N, H_enc)
        encoder_outputs, _ =  pad_packed_sequence(encoder_outputs_packed, batch_first=True) # (N, L_src, H_enc)

        # decoder
        decoder_outputs, _ = self.decoder(tgt_embeddings, encoder_hidden) # decoder_outputs: (N, L_tgt, H_dec)

        # attention
        attention_outputs = self.attention(query = decoder_outputs,
                                           key = encoder_outputs,
                                           mask = attention_mask) # (N, L_tgt, H_enc)

        # output
        cat_outputs = torch.concat([decoder_outputs, attention_outputs], dim=-1) # (N, L_tgt, H_dec + H_enc) 
        logits = self.output(cat_outputs) # (N, L_tgt, vocab_size)
        return logits.permute(0, 2, 1) # (N, vocab_size, L)

Appending to src/model.py


# Training

In [11]:
from src.model import NMTModel
from src.trainer import trainer, save_training_artifacts

torch.manual_seed(42)

# PREPARE TOKENIZER 
tokenizer = tokenizers.Tokenizer.from_file("tokenizer/bpe_tokenizer.json")
vocab_size = tokenizer.get_vocab_size()

# PREPARE DATALOADER
train_ds = NMTDataset(train.source_text, train.target_text)
val_ds = NMTDataset(val.source_text, val.target_text)

train_loader = DataLoader(train_ds, batch_size=hp.batch_size,
                          collate_fn = collate_fn, pin_memory=True, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=hp.batch_size,
                        collate_fn = collate_fn, pin_memory=True, shuffle=False)

# TRAINING
model = NMTModel(vocab_size, **hp.model_hparams).to(device)

optimizer = torch.optim.NAdam(model.parameters(), **hp.optimizer_hparams)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
metric = Accuracy(task="multiclass", num_classes = vocab_size).to(device)

train_logs = trainer(model=model, optimizer=optimizer,
                     loss_fn=loss_fn, metric=metric,
                     device=device, train_loader=train_loader,
                     val_loader=val_loader, **hp.trainer_hparams)

save_training_artifacts(model, train_logs,
                        params_path= os.path.join(MODEL_PATH, "nmt_model_params.pt"),
                        logs_path=os.path.join(ARTIFACTS_PATH, "train_logs.json"))

Epoch 1/15: 100%|███████████████████████████████████████████████████████████████████| 2946/2946 [02:49<00:00, 17.35it/s]


 Epoch 1/15, train_loss: 2.6389, val_loss: 1.9470, val_metric: 0.2292, lr: 0.001, epoch_time: 181.49s


Epoch 2/15: 100%|███████████████████████████████████████████████████████████████████| 2946/2946 [02:48<00:00, 17.46it/s]


 Epoch 2/15, train_loss: 1.6797, val_loss: 1.6846, val_metric: 0.2409, lr: 0.001, epoch_time: 180.05s


Epoch 3/15: 100%|███████████████████████████████████████████████████████████████████| 2946/2946 [02:49<00:00, 17.43it/s]


...

 Epoch 13/15, train_loss: 1.0845, val_loss: 1.4825, val_metric: 0.2516, lr: 0.001, epoch_time: 179.84s


Epoch 14/15: 100%|██████████████████████████████████████████████████████████████████| 2946/2946 [02:48<00:00, 17.47it/s]


 Epoch 14/15, train_loss: 1.0711, val_loss: 1.4781, val_metric: 0.2519, lr: 0.001, epoch_time: 180.05s


Epoch 15/15: 100%|██████████████████████████████████████████████████████████████████| 2946/2946 [02:48<00:00, 17.50it/s]


 Epoch 15/15, train_loss: 1.0568, val_loss: 1.4676, val_metric: 0.2528, lr: 0.001, epoch_time: 179.78s
✅ Model params saved to /mnt/c/Users/ASUS/Documents/Machine-Learning/GitHUB/AttentionNMT/model/nmt_model_params.pt
✅ Logs saved to /mnt/c/Users/ASUS/Documents/Machine-Learning/GitHUB/AttentionNMT/artifacts/train_logs.json


# Inference

In [12]:
%%writefile src/inference.py
import torch, tokenizers

def translate(model, tokenizer, src_text, device, max_len=20, sample=False, temperture=0.7):
    encoded_text = tokenizer.encode(src_text)
    
    src_ids = torch.tensor(encoded_text.ids, dtype=torch.long).reshape(1,-1).to(device)
    src_masks = torch.tensor(encoded_text.attention_mask, dtype=torch.long).reshape(1,-1).to(device)
    tgt_ids = torch.tensor([tokenizer.token_to_id("[BOS]")], dtype=torch.long).reshape(1,-1).to(device)
    
    eos_token_id = tokenizer.token_to_id("[EOS]")
    model.eval()
    
    for _ in range(max_len):
        with torch.no_grad():
            logits = model(src_ids, src_masks, tgt_ids)[:, :, -1] # (1, vocab_size)
            
        if sample:
            scaled_logits = logits / temperture
            probs = torch.softmax(scaled_logits, dim=-1)
            next_token_id = torch.multinomial(probs, num_samples=1)
        else:
            next_token_id = logits.argmax(dim=1, keepdim=True)
                
        tgt_ids = torch.cat([tgt_ids, next_token_id], dim=1)
            
        if next_token_id.item() == eos_token_id:
            break
                
    return tokenizer.decode(tgt_ids.squeeze(0).cpu().numpy())

Overwriting src/inference.py


In [13]:
%%writefile app.py
import torch, tokenizers
import gradio as gr
from src.model import NMTModel
from src.config import HPARAMS
from src.inference import translate
from src.ui import build_demo

### CONFIGURATION ###
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch Device: {device}")
hp = HPARAMS()

### LOAD TOKENIZER ###
tokenizer = tokenizers.Tokenizer.from_file("tokenizer/bpe_tokenizer.json")

### LOAD MODEL ###
model = NMTModel(tokenizer.get_vocab_size(), **hp.model_hparams).to(device)
state_dict = torch.load("model/nmt_model_params.pt", map_location=device, weights_only=True)
model.load_state_dict(state_dict)

### GRADIO APP ###
def translate_fn(src_text, max_len):
    return translate(model, tokenizer, src_text, device, max_len=max_len)

inputs = [
    gr.Textbox(label="English Text", lines=3),
    gr.Slider(10, 100, value=20, step=5, label="Max Length"),
]

outputs = [gr.Textbox(label="Translation", lines=5, interactive=False)]

demo =  build_demo(
    translate_fn,
    inputs,
    outputs,
    english_title = "# 🌐 AttentionNMT: GRU-Attention Encoder-Decoder NMT",
    persian_title = "# 🌐 ترجمه‌ی ماشینی دوزبانه با معماری رمزگذار–رمزگشا و Attention",
    assets_dir = "assets",
    app_title = "AttentionNMT"
)

demo.launch()

Overwriting app.py


In [14]:
!python app.py

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Torch Device: cuda
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
^C
Keyboard interruption in main thread... closing server.
